# ДЗ-14: RAG-система с поиском по собственной базе документов (Milvus + OpenRouter)

Реализация векторного поиска для **RAG** (Retrieval-Augmented Generation) на векторной БД **Milvus**,
поднятой в Docker. LLM-провайдер — **OpenRouter** (OpenAI-совместимый API, единый ключ на все модели).
Эмбеддинги — `openai/text-embedding-3-small`, генерация ответа — `openai/gpt-4o-mini`. Обе модели
платные, но очень дёшевы: эмбеддинги ~$0.02 за 1M токенов, чат ~$0.15 / $0.60 за 1M input/output
токенов — цена та же, что и напрямую у OpenAI, OpenRouter просто проксирует запрос.

## Что внутри
**Часть 1. Настройка и индексация**
- подключение к Milvus (запущен через `docker compose`);
- генерация эмбеддингов через OpenRouter;
- проектирование схемы коллекции и индексация документов.

**Изучение ANN-алгоритмов** — сравнение **IVF vs HNSW vs ANNOY**:
trade-off «скорость ↔ точность» на замерах recall и latency.

**Часть 2. Реализация поиска**
- семантический поиск по векторам;
- метрики похожести (cosine / L2 / IP);
- фильтрация по метаданным;
- подбор `top-k` и параметров поиска.

> ⚠️ **Важно про ANNOY.** Milvus **удалил** индекс ANNOY начиная с версии 2.3.0
> ([issue #30608](https://github.com/milvus-io/milvus/issues/30608)). В актуальном Milvus 2.5 нативно
> доступны **IVF** (IVF_FLAT/IVF_SQ8/IVF_PQ), **HNSW**, SCANN, DISKANN. Поэтому IVF и HNSW мы
> сравниваем прямо в Milvus, а ANNOY — через оригинальную библиотеку `annoy` (Spotify) отдельным
> блоком: алгоритм и его trade-offs от этого не меняются.

## Шаг 0. Поднять Milvus в Docker

В папке `hw14/` выполни в терминале:

```bash
docker compose up -d        # поднимет etcd + minio + milvus-standalone
docker compose ps           # дождись, пока milvus-standalone станет healthy (~30-60 сек)
```

Затем установи зависимости и положи ключ OpenRouter в `.env`:

```bash
pip install -r requirements.txt
copy .env.example .env       # Windows;  на macOS/Linux: cp .env.example .env
#  -> впиши OPENROUTER_API_KEY в .env (получить на https://openrouter.ai/keys)
```

In [ ]:
import os, time, json, random
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pymilvus import (
    connections, utility,
    FieldSchema, CollectionSchema, DataType, Collection,
)

load_dotenv()

MILVUS_HOST = os.getenv("MILVUS_HOST", "localhost")
MILVUS_PORT = os.getenv("MILVUS_PORT", "19530")

# ── Что такое эмбеддинг ───────────────────────────────────────────────────────
# Эмбеддинг — это представление текста в виде плотного вектора чисел фиксированной
# длины. Модель обучена так, что семантически близкие тексты дают близкие векторы
# (по косинусному расстоянию). Именно по этим векторам и работает векторный поиск:
# вопрос пользователя превращаем в вектор и ищем ближайшие векторы-документы.
#
# ── Какие эмбеддинги можно использовать ──────────────────────────────────────
#  OpenRouter (облако, платно, единый OpenAI-совместимый API на все модели):
#     • openai/text-embedding-3-small  — dim 1536, ~$0.02/1M ток., отличный baseline  ← берём его
#     • openai/text-embedding-3-large  — dim 3072, точнее, но дороже
#     • openai/text-embedding-ada-002  — dim 1536, легаси, заметно слабее 3-*
#  Локальные / open-source (бесплатно, без интернета, через sentence-transformers):
#     • all-MiniLM-L6-v2        — dim 384,  быстрый, лёгкий
#     • BAAI/bge-m3, intfloat/e5-large — топовое качество, мультиязычность
#  Выбор влияет на: размерность вектора (dim в схеме Milvus), качество поиска,
#  цену и скорость. Метрику почти всегда берут COSINE.
EMBED_MODEL = "openai/text-embedding-3-small"
EMBED_DIM   = 1536                      # размерность text-embedding-3-small
CHAT_MODEL  = "openai/gpt-4o-mini"      # для генерации ответа в мини-RAG (дёшево: $0.15/$0.60 за 1M ток.)

# OpenRouter — OpenAI-совместимый API: тот же клиент, меняются только base_url и ключ.
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
print("OpenRouter client готов. Модель эмбеддингов:", EMBED_MODEL, "| модель чата:", CHAT_MODEL)

In [ ]:
# Подключаемся к Milvus и проверяем связь
connections.connect(alias="default", host=MILVUS_HOST, port=MILVUS_PORT)
print("Подключено к Milvus", utility.get_server_version())
print("Существующие коллекции:", utility.list_collections())

## Часть 1. Эмбеддинги и индексация

### 1.1 Генерация эмбеддингов через OpenRouter

In [ ]:
def embed(texts: list[str]) -> list[list[float]]:
    """Превращает список строк в список векторов через OpenRouter (модель EMBED_MODEL).
    Один батч-запрос на весь список — так дешевле и быстрее, чем по одному."""
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in resp.data]

# быстрый sanity-check
v = embed(["проверка эмбеддинга"])[0]
print("Размерность вектора:", len(v), "| первые 5 чисел:", [round(x, 4) for x in v[:5]])

In [ ]:
# Загружаем собственный датасет документов с метаданными
with open("documents.json", encoding="utf-8") as f:
    docs = json.load(f)

print(f"Документов в базе: {len(docs)}")
print("Категории:", sorted({d["category"] for d in docs}))

texts = [d["text"] for d in docs]
vectors = embed(texts)                       # эмбеддинги всех документов (один запрос)
print("Сгенерировано векторов:", len(vectors), "| dim:", len(vectors[0]))

### 1.2 Оптимальная схема коллекции

Коллекция в Milvus = аналог таблицы. Поля бывают **векторные** (по ним ANN-поиск) и
**скалярные** (по ним фильтрация). Ключевые решения при проектировании схемы:

| Поле | Тип | Зачем |
|---|---|---|
| `id` | INT64, primary, `auto_id=True` | первичный ключ, Milvus сам генерит |
| `vector` | FLOAT_VECTOR, `dim=1536` | сам эмбеддинг; `dim` обязан совпадать с моделью |
| `text` | VARCHAR(2000) | исходный текст — вернём его как результат поиска |
| `category`, `source` | VARCHAR | скалярные поля для **фильтрации по метаданным** |
| `year` | INT64 | числовой фильтр (диапазоны `year >= 2023`) |

`max_length` у VARCHAR ограничивает размер строки; для `text` берём с запасом.

In [ ]:
COLLECTION = "rag_docs"

def recreate_collection(name: str, dim: int) -> Collection:
    """Создаёт коллекцию с нуля (если была — удаляет). Удобно при повторных прогонах."""
    if utility.has_collection(name):
        utility.drop_collection(name)

    fields = [
        FieldSchema(name="id",       dtype=DataType.INT64,        is_primary=True, auto_id=True),
        FieldSchema(name="vector",   dtype=DataType.FLOAT_VECTOR, dim=dim),
        FieldSchema(name="text",     dtype=DataType.VARCHAR,      max_length=2000),
        FieldSchema(name="category", dtype=DataType.VARCHAR,      max_length=64),
        FieldSchema(name="source",   dtype=DataType.VARCHAR,      max_length=64),
        FieldSchema(name="year",     dtype=DataType.INT64),
    ]
    schema = CollectionSchema(fields, description="RAG: документы + метаданные")
    return Collection(name=name, schema=schema)

col = recreate_collection(COLLECTION, EMBED_DIM)
print("Создана коллекция:", col.name)
print("Поля:", [f.name for f in col.schema.fields])

In [ ]:
# Вставляем данные. Row-based формат: список словарей (id не передаём — auto_id).
rows = [
    {"vector": vectors[i], "text": d["text"], "category": d["category"],
     "source": d["source"], "year": d["year"]}
    for i, d in enumerate(docs)
]
col.insert(rows)
col.flush()                                  # сбрасываем на диск
print("Вставлено сущностей:", col.num_entities)

### 1.3 Индексы: IVF vs HNSW (параметры и trade-offs)

Без индекса Milvus делает полный перебор (FLAT) — точно, но медленно на больших объёмах.
**ANN-индекс** ускоряет поиск ценой небольшой потери точности.

**IVF_FLAT** — кластеризует векторы на `nlist` ячеек (k-means). При поиске смотрит только
`nprobe` ближайших ячеек.
- `nlist` (build): больше ячеек → быстрее, но риск пропустить соседей на границах.
- `nprobe` (search): больше → точнее, но медленнее. Главный «руль» точность↔скорость.

**HNSW** — многослойный граф ближайших соседей (навигация как «по skip-list»).
- `M` (build): число рёбер на узел; больше → точнее и больше памяти.
- `efConstruction` (build): тщательность построения графа.
- `ef` (search): ширина луча поиска; больше → точнее и медленнее. Главный «руль».

Эмпирика: **HNSW** обычно быстрее и точнее на запрос, но дольше строится и ест больше RAM.
**IVF** дешевле по памяти и быстрее строится — выгоден на очень больших коллекциях.

In [ ]:
def build_index(collection: Collection, index_type: str, metric: str = "COSINE", params: dict | None = None):
    """Создаёт векторный индекс заданного типа и загружает коллекцию в память для поиска."""
    defaults = {
        "IVF_FLAT": {"nlist": 128},
        "HNSW":     {"M": 16, "efConstruction": 200},
        "FLAT":     {},
    }
    collection.release()
    if collection.has_index():
        collection.drop_index()
    t0 = time.perf_counter()
    collection.create_index(
        field_name="vector",
        index_params={"index_type": index_type, "metric_type": metric,
                      "params": params or defaults[index_type]},
    )
    collection.load()                        # без load() поиск невозможен
    return time.perf_counter() - t0

build_s = build_index(col, "HNSW", metric="COSINE")
print(f"Индекс HNSW построен и загружен за {build_s:.2f} c")

### Метрики похожести (similarity metrics)

Milvus поддерживает для float-векторов:
- **COSINE** — косинусная близость (угол между векторами, без учёта длины). Дефолт для текста.
- **L2** — евклидово расстояние (учитывает длину). Меньше = ближе.
- **IP** (inner product) — скалярное произведение; на L2-нормированных векторах эквивалентно cosine.

Метрика задаётся при создании индекса и **должна совпадать** в индексе и в запросе.
Для текстовых эмбеддингов берём **COSINE**: у Milvus при ней `distance` = похожесть (больше = лучше).

## Часть 2. Реализация поиска

### 2.1 Семантический поиск + top-k

In [ ]:
def search(query: str, top_k: int = 3, expr: str | None = None, ef: int = 64):
    """Семантический поиск: текст запроса -> вектор -> ближайшие документы.
    expr — необязательный фильтр по метаданным (булево выражение Milvus)."""
    qv = embed([query])[0]
    results = col.search(
        data=[qv],
        anns_field="vector",
        param={"metric_type": "COSINE", "params": {"ef": ef}},  # ef — для HNSW
        limit=top_k,
        expr=expr,                                              # None = без фильтра
        output_fields=["text", "category", "source", "year"],
    )
    hits = results[0]
    return [
        {"score": round(h.distance, 4), "category": h.entity.get("category"),
         "year": h.entity.get("year"), "text": h.entity.get("text")}
        for h in hits
    ]

for r in search("как работает механизм внимания в нейросетях?", top_k=3):
    print(f"[{r['score']}] ({r['category']}, {r['year']}) {r['text'][:90]}...")

In [ ]:
# top-k: сколько документов вернуть. Для RAG обычно 3-5 — это контекст для LLM.
print("=== top_k = 5, запрос про векторные БД ===")
for r in search("какие бывают векторные базы данных и индексы", top_k=5):
    print(f"[{r['score']}] ({r['category']}) {r['text'][:80]}...")

### 2.2 Фильтрация по метаданным

Скалярные поля позволяют сузить поиск **до** ANN-этапа. Синтаксис выражений Milvus:
`==`, `!=`, `>`, `>=`, `in [...]`, `&&`, `||`, `like "..."`.

In [ ]:
q = "современные подходы в машинном обучении"

print("--- Без фильтра ---")
for r in search(q, top_k=3):
    print(f"[{r['score']}] ({r['category']}, {r['year']}) {r['text'][:70]}...")

print("\n--- Только category == 'ml' ---")
for r in search(q, top_k=3, expr='category == "ml"'):
    print(f"[{r['score']}] ({r['category']}, {r['year']}) {r['text'][:70]}...")

print("\n--- Свежее: year >= 2023 И источник не paper ---")
for r in search(q, top_k=3, expr='year >= 2023 && source != "paper"'):
    print(f"[{r['score']}] ({r['category']}, {r['year']}) {r['text'][:70]}...")

### 2.3 Мини-RAG: retrieval + генерация ответа

Собираем найденные документы в контекст и просим LLM ответить **только по нему** — это и есть
суть RAG: меньше галлюцинаций, ответ опирается на твою базу.

In [ ]:
def rag_answer(question: str, top_k: int = 3) -> str:
    found = search(question, top_k=top_k)
    context = "\n".join(f"- {r['text']}" for r in found)
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": "Отвечай кратко и ТОЛЬКО на основе контекста. "
                                           "Если ответа в контексте нет — так и скажи."},
            {"role": "user", "content": f"Контекст:\n{context}\n\nВопрос: {question}"},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content

print(rag_answer("Чем HNSW отличается от IVF и за что отвечает параметр nprobe?"))

## Изучение ANN-алгоритмов: IVF vs HNSW vs ANNOY

### Методология бенчмарка
На 20 реальных документах разница индексов незаметна, поэтому для честного сравнения
**скорость ↔ точность** генерируем синтетический набор: `N` случайных векторов размерности `D`.
OpenRouter здесь **не вызываем** (вектора случайные → бесплатно).

Метрики:
- **Recall@k** — доля истинных ближайших соседей, которые индекс реально нашёл (1.0 = идеально).
  Эталон («ground truth») считаем полным перебором (FLAT) на numpy.
- **Latency** — среднее время одного запроса, мс.

In [ ]:
# Синтетический набор для бенчмарка
np.random.seed(42)
N_BASE   = 20000      # сколько векторов в индексе
N_QUERY  = 100        # сколько поисковых запросов усредняем
D        = 128        # размерность (любая; меньше 1536 — чтобы бенчмарк был быстрым)
TOPK     = 10

def l2norm(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)

base   = l2norm(np.random.randn(N_BASE,  D).astype("float32"))
query  = l2norm(np.random.randn(N_QUERY, D).astype("float32"))

# Ground truth: точные top-k по косинусу (на нормированных векторах = скалярное произведение)
sims = query @ base.T                          # (N_QUERY, N_BASE)
gt   = np.argsort(-sims, axis=1)[:, :TOPK]      # индексы истинных соседей
print(f"База: {N_BASE} векторов x {D} dim | запросов: {N_QUERY} | top-k: {TOPK}")

def recall_at_k(found_ids: np.ndarray) -> float:
    """Доля пересечения найденных id с истинными, усреднённая по запросам."""
    hits = [len(set(found_ids[i]) & set(gt[i])) for i in range(len(gt))]
    return sum(hits) / (len(gt) * TOPK)

In [ ]:
# Готовим отдельную коллекцию Milvus под синтетику
BENCH = "ann_benchmark"
if utility.has_collection(BENCH):
    utility.drop_collection(BENCH)

bfields = [
    FieldSchema(name="id",     dtype=DataType.INT64,        is_primary=True, auto_id=False),
    FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=D),
]
bench = Collection(BENCH, CollectionSchema(bfields, description="синтетика для ANN-бенчмарка"))
bench.insert([{"id": i, "vector": base[i].tolist()} for i in range(N_BASE)])
bench.flush()
print("Вставлено в bench:", bench.num_entities)

In [ ]:
def bench_milvus(index_type: str, build_params: dict, search_params_list: list[dict]) -> list[dict]:
    """Строит индекс в Milvus и гоняет поиск с разными search-параметрами."""
    bench.release()
    if bench.has_index():
        bench.drop_index()
    t0 = time.perf_counter()
    bench.create_index("vector", {"index_type": index_type, "metric_type": "IP",
                                   "params": build_params})
    bench.load()
    build_t = time.perf_counter() - t0

    rows = []
    qlist = query.tolist()
    for sp in search_params_list:
        # прогрев
        bench.search(qlist[:5], "vector", {"metric_type": "IP", "params": sp}, limit=TOPK)
        t0 = time.perf_counter()
        res = bench.search(qlist, "vector", {"metric_type": "IP", "params": sp}, limit=TOPK)
        dt = (time.perf_counter() - t0) / N_QUERY * 1000
        found = np.array([[h.id for h in res[i]] for i in range(N_QUERY)])
        rows.append({"algo": index_type, "params": str(sp), "build_s": round(build_t, 2),
                     "recall@10": round(recall_at_k(found), 3), "latency_ms": round(dt, 3)})
    return rows

ivf_rows  = bench_milvus("IVF_FLAT", {"nlist": 256},
                         [{"nprobe": n} for n in (1, 4, 16, 64, 256)])
hnsw_rows = bench_milvus("HNSW", {"M": 16, "efConstruction": 200},
                         [{"ef": e} for e in (16, 32, 64, 128, 256)])
pd.DataFrame(ivf_rows + hnsw_rows)

In [ ]:
# ANNOY — через оригинальную библиотеку Spotify (в Milvus 2.3+ его нет).
from annoy import AnnoyIndex

def bench_annoy(n_trees_list: list[int], search_k_list: list[int]) -> list[dict]:
    rows = []
    for n_trees in n_trees_list:
        t0 = time.perf_counter()
        idx = AnnoyIndex(D, "angular")          # angular ~ косинусное расстояние
        for i in range(N_BASE):
            idx.add_item(i, base[i])
        idx.build(n_trees)
        build_t = time.perf_counter() - t0
        for search_k in search_k_list:
            # прогрев
            for q in query[:5]:
                idx.get_nns_by_vector(q, TOPK, search_k=search_k)
            t0 = time.perf_counter()
            found = np.array([idx.get_nns_by_vector(q, TOPK, search_k=search_k) for q in query])
            dt = (time.perf_counter() - t0) / N_QUERY * 1000
            rows.append({"algo": "ANNOY", "params": f"n_trees={n_trees}, search_k={search_k}",
                         "build_s": round(build_t, 2),
                         "recall@10": round(recall_at_k(found), 3), "latency_ms": round(dt, 3)})
    return rows

annoy_rows = bench_annoy(n_trees_list=[10, 50], search_k_list=[100, 1000, 10000])
pd.DataFrame(annoy_rows)

In [ ]:
# Сводная таблица: точность vs скорость по всем трём алгоритмам
allrows = ivf_rows + hnsw_rows + annoy_rows
df = pd.DataFrame(allrows).sort_values(["algo", "recall@10"]).reset_index(drop=True)
print(df.to_string(index=False))

print("\nЛучшая latency при recall >= 0.95:")
good = df[df["recall@10"] >= 0.95].sort_values("latency_ms")
print(good.groupby("algo").first()[["params", "recall@10", "latency_ms", "build_s"]].to_string())

## Выводы: trade-offs «скорость ↔ точность»

| Алгоритм | Главный «руль» точности | Сильная сторона | Слабая сторона |
|---|---|---|---|
| **IVF_FLAT** | `nprobe` (search), `nlist` (build) | быстро строится, экономит RAM | при низком `nprobe` теряет соседей на границах кластеров |
| **HNSW** | `ef` (search), `M`/`efConstruction` (build) | лучший recall/latency на запрос | долгое построение, высокий расход RAM |
| **ANNOY** | `search_k`, `n_trees` (build) | простой, файловый индекс (mmap), хорош для read-only | recall ниже HNSW, индекс надо перестраивать при добавлении данных |

**Практические правила:**
- Главный регулятор точности — это **search-параметр** (`nprobe` / `ef` / `search_k`): крутишь вверх →
  recall растёт, latency растёт. Build-параметры (`nlist` / `M` / `n_trees`) задают «потолок».
- Для большинства RAG-задач в Milvus берём **HNSW + COSINE**: лучший баланс на запрос.
- **IVF** выгоден при очень больших коллекциях и ограниченной памяти.
- **ANNOY** хорош, когда индекс строится один раз и раздаётся read-only (например, в мобильном/edge).
- `top-k` для RAG держим небольшим (3–5) — это контекст для LLM, а не «найти всё».

In [ ]:
# Прибираемся: освобождаем память (коллекции при желании можно и удалить)
col.release()
bench.release()
# utility.drop_collection("ann_benchmark")   # раскомментируй, чтобы удалить синтетику
print("Готово. Активные коллекции:", utility.list_collections())